# with_steps Orchestration Demo

Demonstrates linear orchestration with batching, parallelism, conditionals, retries, waits, and timeouts using `run_steps` / `StepSpec`.

In [2]:
# Runtime OpenAI key setup (fallback to stubbed LLM)
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (press Enter to skip): ")

from agentic_codex import AgentBuilder, Context, FunctionAdapter, EnvOpenAIAdapter, StepSpec, run_steps
from agentic_codex.core.schemas import AgentStep, Message

def stub_llm(prompt: str) -> str:
    lines = [ln.strip() for ln in prompt.splitlines() if ln.strip()]
    return f"[stub llm] {lines[-1][:120] if lines else 'response'}"

try:
    llm_adapter = EnvOpenAIAdapter(model="gpt-4o-mini")
except Exception:
    llm_adapter = FunctionAdapter(stub_llm)

def make_agent(name: str, role: str, tag: str):
    def step(ctx: Context) -> AgentStep:
        note = ctx.scratch.get("batch_item", tag)
        content = f"{tag}:{note}:{ctx.goal}"
        return AgentStep(out_messages=[Message(role=name, content=content)], state_updates={"last": content})
    return (
        AgentBuilder(name=name, role=role)
        .with_llm(llm_adapter)
        .with_step(step)
        .build()
    )

planner = make_agent("planner", "planner", "plan")
worker = make_agent("worker", "worker", "work")
reviewer = make_agent("reviewer", "qa", "review")

ctx = Context(goal="Ship feature X", scratch={"tasks": ["design", "impl", "test"], "ok_to_ship": True})

steps = [
    StepSpec(name="plan", fn=planner.run),
    StepSpec(name="do_batch", fn=worker.run, batch_key="tasks", parallel=True, isolate_state=True, retry=1, timeout=2.0),
    StepSpec(name="wait-a-moment", fn=worker.run, wait_seconds=0.1),
    StepSpec(name="review", fn=reviewer.run, condition=lambda c: c.scratch.get("ok_to_ship", False)),
]

results = run_steps(steps, ctx, max_parallel=4)
[m.content for r in results for m in r.out_messages]

['plan:plan:Ship feature X',
 'work:design:Ship feature X',
 'work:impl:Ship feature X',
 'work:test:Ship feature X',
 'work:work:Ship feature X',
 'review:review:Ship feature X']